### Util Funcs

In [1]:
import math
import numpy as np

def z_from_confidence(conf_level: float) -> float:
    """
    Return two-sided Z critical for a (1 - alpha) confidence level.
    Supports common levels without external libraries.
    """
    common = {
        0.80: 1.2816,
        0.85: 1.4395,
        0.90: 1.6449,
        0.95: 1.9600,
        0.98: 2.3263,
        0.99: 2.5758,
        0.999: 3.2905,
    }
    if conf_level in common:
        return common[conf_level]
    raise ValueError(
        f"Unsupported confidence level {conf_level}. "
        "Pick one of: " + ", ".join(map(str, sorted(common.keys())))
    )

def precision_sample_size(E: float,
                          sigma: float = 0.5,
                          conf_level: float = 0.95,
                          z: float | None = None,
                          ceil: bool = True) -> float | int:
    """
    Precision-based sample size for estimating a mean score with desired
    half-width (margin of error) E:
        n = (Z_(alpha/2) * sigma / E)^2

    Args
    ----
    E : desired half-width of the CI (margin of error), e.g. 0.05
    sigma : population SD of item scores (use pilot estimate).
            For 0–1 bounded scores, sigma <= 0.5.
    conf_level : two-sided confidence level, e.g. 0.95 (ignored if `z` given).
    z : optional Z critical value. If provided, overrides `conf_level`.
    ceil : if True, return math.ceil(n) as an integer (recommended for planning).

    Returns
    -------
    n : required number of independent items (samples). Integer if ceil=True.
    """
    if E <= 0:
        raise ValueError("E must be > 0.")
    if sigma <= 0:
        raise ValueError("sigma must be > 0.")
    Z = z if z is not None else z_from_confidence(conf_level)
    n = (Z * sigma / E) ** 2
    return math.ceil(n) if ceil else n

# --- Optional helpers for 0–1 bounded scores (e.g., rubric percentages) ---

def sigma_from_proportion(p: float) -> float:
    """
    For binary/0–1 items, SD = sqrt(p*(1-p)).
    Use pilot estimate of success rate `p`. If unknown, use p=0.5 (worst case).
    """
    if not 0 <= p <= 1:
        raise ValueError("p must be between 0 and 1.")
    return math.sqrt(p * (1 - p))

### Example 1

In [2]:
llm_as_a_judge_scores = [0.2, 0.3, 0.2, 0.4, 0.1, 0.2, 0.4, 0.3, 0.2, 0.3, 
                         0.2, 0.4, 0.1, 0.2, 0.4, 0.3, 0.1, 0.2, 0.3, 0.3]

In [3]:
sigma_hat = np.std(llm_as_a_judge_scores)
print(f"Estimated Standard Deviation: {sigma_hat}")

sample_size = precision_sample_size(E=0.05, sigma=sigma_hat, conf_level=0.95)
print(f"Recommended sample size: {sample_size}")

Estimated Standard Deviation: 0.09733961166965893
Recommended sample size: 15


### Example 2

In [4]:
llm_as_a_judge_scores = [0.5, 0.3, 0.2, 0.4, 0.1, 0.2, 0.6, 0.3, 0.1, 0.2, 
                         0.2, 0.6, 0.1, 0.2, 0.4, 0.3, 0.5, 0.7, 0.9, 0.1]

In [5]:
sigma_hat = np.std(llm_as_a_judge_scores)
print(f"Estimated Standard Deviation: {sigma_hat}")

sample_size = precision_sample_size(E=0.05, sigma=sigma_hat, conf_level=0.95)
print(f"Recommended sample size: {sample_size}")

Estimated Standard Deviation: 0.22017038856303994
Recommended sample size: 75
